In [17]:
import anthropic
from dotenv import load_dotenv
import os
import json
from datetime import datetime
load_dotenv()
client = anthropic.Anthropic()
MUSIC_FILE = "music_storage.json"
messages=[]


tools=[{"name":"add_music","description":"Add a song or album to my records and give it a rating, and a short description",        
        "input_schema":{
        "type": "object",
        "properties": {
            "name":{"type":"string","description":"name of the album or song"},
            "artist":{"type":"string","descripton":"The artist of the song or album"},
            "rating": {"type": "number", "description": "Rating of the song or album being added"},
            "genre": {"type": "string", "description": "Genre of the album or song"},
            "category": {"type": "string", "description": "Is it a album or song"},
            "description": {"type": "string", "description": "My description of the song/album"},        },
            "required":["rating","genre","category","description"]
        }},
      {"name":"list_music","description":"Find a list of songs and albums of a certain or all genres",        
        "input_schema":{
        "type": "object",
        "properties": {
            "genre": {"type": "string", "description": "All albums and songs of a certain genre or if no parameter, all genres"},
        },
            "required":[]
        }},
      {"name":"list_music_by_artist","description":"Find a list of songs and albums of a certain artist",        
        "input_schema":{
        "type": "object",
        "properties": {
            "artist": {"type": "string", "description": "All albums and songs of a certain artist"},
        },
            "required":[]
        }},
      {"name":"delete_music","description":"Delete song or album of certain id",        
        "input_schema":{
        "type": "object",
        "properties": {
            "music_id": {"type": "number", "description": "Id of the song/album you want to delete"},
        },
            "required":["transaction_id"]
        }}
]

def load_music():
    if os.path.exists(MUSIC_FILE):
        with open(MUSIC_FILE, 'r') as f:
            transactions = json.load(f)
        return transactions
    else:
        return []

def save_music(music):
    with open(MUSIC_FILE, 'w') as f:
        json.dump(music,f,indent=2)

def add_music(name,rating, artist, genre, category, description):
    data = load_music()
    highest = 0
    for i in data:
        if highest<i["id"]:
            highest=i["id"]
    new_id = highest + 1
    transaction = {
        "id":new_id,
        "name":name,
        "artist":artist,
        "rating":rating,
        "genre":genre,
        "category":category,
        "description":description,
    }
    data.append(transaction)
    save_music(data)
    return transaction

def list_music(genre=None):
    data = load_music()
    if genre is None:
        return data
    returntrans = []
    for i in data:
        if i["genre"].lower()==genre.lower():
            returntrans.append(i)
    return returntrans

def list_music_by_artist(artist=None):
    data = load_music()
    if artist is None:
        return data
    returntrans = []
    for i in data:
        if i["artist"].lower()==artist.lower():
            returntrans.append(i)
    return returntrans

def delete_music(transaction_id):
    data = load_music()
    deleted = None
    kept = []
    for i in data:
        if i["id"] == transaction_id:
            deleted = i
        else:
            kept.append(i)
    if deleted is None:
        return{"Error": "No object found on this id for deletion"}
    save_transactions(kept)
    return deleted

tool_functions = {"add_music":add_music,"list_music":list_music,"list_music_by_artist":list_music_by_artist,"delete_music":delete_music}
while True:
    user_text = input()
    if user_text == "exit":
        break
    messages.append({"role":"user","content":user_text})
    message = client.messages.create(
        tools=tools,
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=messages
    )
    tool_results = []
    messages.append({"role":"assistant","content":message.content})
    while message.stop_reason == "tool_use":
        for i in message.content:
            if i.type == "tool_use":
                result = tool_functions[i.name](**i.input)
                tool_results.append({
                    "type":"tool_result",
                    "tool_use_id":i.id,
                    "content":json.dumps(result)
                })
        messages.append({"role":"user","content":tool_results})
        message = client.messages.create(
            tools=tools,
            model="claude-haiku-4-5-20251001",
            max_tokens=1024,
            messages=messages
        )
        messages.append({"role": "assistant", "content": message.content})
        
    for block in message.content:
        if block.type == "text":
            print(block.text)


 exit
